# RSU-Tier Trust Score Analyzer (Approach 2)

NS-3 SDVEN Sybil-attack simulation — **RSU-tier** Trust Analyzer (thesis §3.4.3-3.4.5,
`sybil-attack/docs/Sybil_attack_project.pdf`)

**Companion notebook.** `vehicle_trust_score_model.ipynb` (Approach 1) trains the
**vehicle-tier** analyzer: one FL client per vehicle, aggregated in two hops
(vehicle &rarr; RSU &rarr; SDN, Eqs 3.27-3.28), using only what a single observing
vehicle sees. This notebook is **Approach 2**: one FL client **per RSU**, aggregated
in a single hop directly to the SDN/global controller, using what *every* vehicle
reporting through that RSU sees, pooled together. Pooling reporters is exactly
Tier 2 in the thesis (§3.4.3, Eq 3.20: `c_r = [ŷ_i ∥ φ_meta(v_i)]` for `v_i ∈ V_r`) —
here it is implemented as a numeric feature/MLP pipeline instead of an LLM agent,
so it can be trained and evaluated the same way as Approach 1.

**Why a second analyzer.** A single vehicle can only ever report its own local
view of a claimed identity. An RSU, by construction, receives that same claimed
identity's beacons *relayed by every vehicle in range* — so it can see things no
single vehicle can: whether several independent reporters agree on who a claimed
identity really is, whether one vehicle's self-report contradicts what everyone
else says about it, and whether its reported position is consistent across
witnesses. §3 below turns each of those into a trust feature. This is the concrete
sense in which "RSU-level observations... improve the trust score calculation."

**On the dataset — same caveat as Approach 1.** This notebook runs against the same
`outputs/evaluation_runs/` v10-v40 sweep, for the same reason: it is what exists
today, not the intended final dataset (see Approach 1's mismatch #4). Every number
below is a pipeline-correctness check, not a final trust-score result — re-read
Approach 1's summary (§16 there) before quoting any MCC value from this notebook.

**Relationship to the FL equations.** Eq 3.26 (global objective) is unchanged.
Eq 3.27 (mid-tier RSU aggregation of vehicle-client updates) has no role here,
because the client itself now *is* the RSU — there is no vehicle sub-tier to
aggregate. Eq 3.28 (global trimmed-mean aggregation) becomes the *only*
aggregation step, applied directly to RSU-client updates. §9 below states this
explicitly.


In [ ]:
import json
import math
import re
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

from sklearn.metrics import (
    matthews_corrcoef, precision_score, recall_score, f1_score,
    accuracy_score, roc_curve, auc, precision_recall_curve, confusion_matrix,
)

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid')

RNG_SEED = 7
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
rng = np.random.default_rng(RNG_SEED)

EVAL_RUNS_DIR = Path('../outputs/evaluation_runs')
EXPORT_DIR = Path('../fl/trust_analyzer')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Looking for run folders under: {EVAL_RUNS_DIR.resolve()}')


## 1. Discover runs (RSU-tier requires a different log to be present)

Same run-folder naming convention as Approach 1 (`type{N}_<name>_pct{P}_v{V}_r{R}`),
but a run is only usable here if `rsu_vehicle_observation_rows_log.csv` is present
and non-empty — that log, not `vehicle_neighbor_table_log.csv`, is the RSU's pooled
view across all reporting vehicles.


In [ ]:
RUN_NAME_RE = re.compile(
    r"^type(?P<attack_type>\d+)_(?P<attack_name>.+?)_pct(?P<pct>\d+)_v(?P<vehicles>\d+)_r(?P<replicate>\d+)$"
)

RSU_OBS_LOG = "rsu_vehicle_observation_rows_log.csv"


@dataclass
class RunInfo:
    path: Path
    run_id: str
    attack_type: int
    attack_name: str
    attack_percentage: int
    vehicle_count: int
    replicate: int


def discover_runs(root: Path) -> List[RunInfo]:
    found = []
    for child in sorted(root.iterdir()):
        if not child.is_dir():
            continue
        m = RUN_NAME_RE.match(child.name)
        if not m:
            continue
        obs_path = child / RSU_OBS_LOG
        if not obs_path.exists() or obs_path.stat().st_size < 100:
            continue  # missing or header-only / empty
        found.append(RunInfo(
            path=child,
            run_id=child.name,
            attack_type=int(m.group('attack_type')),
            attack_name=m.group('attack_name'),
            attack_percentage=int(m.group('pct')),
            vehicle_count=int(m.group('vehicles')),
            replicate=int(m.group('replicate')),
        ))
    return found


runs = discover_runs(EVAL_RUNS_DIR)
print(f'Discovered {len(runs)} usable run folders (non-empty {RSU_OBS_LOG}).')
runs_df = pd.DataFrame([r.__dict__ for r in runs])
display(runs_df.groupby(['attack_type', 'attack_name', 'attack_percentage']).size()
        .rename('n_run_folders').reset_index())


## 2. Load & label — RSU-tier pooled observation rows

`rsu_vehicle_observation_rows_log.csv` is keyed by `(rsu_id, observed_claimed_id)`
and has one row per *reporting vehicle sighting*, not per receiving vehicle — i.e.
it is already pooled across every `reported_by_vehicle_id` that told this RSU
about this claimed identity. Two row types:
- `self_report` — a vehicle reporting its own BSM (`reported_by_vehicle_id ==
  observed_real_id`, so `is_sybil` is always 0 for these rows by construction).
- `third_party` — a vehicle reporting a *neighbor's* claimed identity — this is
  where Sybil evidence lives, and where independent reporters can corroborate or
  contradict each other.

**Schema note (checked across all 79 usable runs):** 43 runs include
`rssi_estimated_distance_m` (the RSU's own RF-based distance estimate to a
claimed identity, independent of what the vehicle self-reports); 36 do not. Both
schemas are handled by reindexing to a fixed column set and filling absent
columns with `NaN` — features derived from `rssi_estimated_distance_m` (§4, T_RSSI)
degrade gracefully to a neutral value when unavailable rather than erroring.
Ground truth label reuses the same rule as Approach 1:
`observed_real_id != observed_claimed_id`.


In [ ]:
RSU_OBS_COLS = {
    "time", "event", "rsu_id", "observed_claimed_id", "row_type",
    "reported_by_vehicle_id", "observed_real_id", "report_receive_time",
    "observation_time", "bsm_x", "bsm_y", "bsm_z", "bsm_speed", "bsm_heading",
    "estimated_distance", "received_beacon_count", "suspicion_flags",
    "rssi_estimated_distance_m", "dirty", "rows_for_claimed_id", "trigger_seq", "status",
}
SUSPICION_INVALID_V2V_SIGNATURE = 1 << 9  # bit 512, per docs/DETECTION_FLAGS_REPORT.md


def load_run(run: RunInfo) -> Optional[pd.DataFrame]:
    path = run.path / RSU_OBS_LOG
    df = pd.read_csv(path, usecols=lambda c: c in RSU_OBS_COLS)
    if df.empty:
        return None
    for col in RSU_OBS_COLS:
        if col not in df.columns:
            df[col] = np.nan  # e.g. rssi_estimated_distance_m on the 36 runs without it

    df = df.sort_values(["rsu_id", "observed_claimed_id", "time"]).reset_index(drop=True)
    df["is_sybil"] = (df["observed_real_id"] != df["observed_claimed_id"]).astype(int)
    df["token_valid"] = (
        (df["suspicion_flags"].fillna(0).astype(int) & SUSPICION_INVALID_V2V_SIGNATURE) == 0
    ).astype(int)
    df["controller_id"] = 0  # single global controller in this sweep, same mismatch as Approach 1 §2

    df["run_id"] = run.run_id
    df["attack_type"] = run.attack_type
    df["attack_name"] = run.attack_name
    df["attack_percentage"] = run.attack_percentage
    df["vehicle_count"] = run.vehicle_count
    return df


raw_frames = []
skipped = []
for r in runs:
    frame = load_run(r)
    if frame is None:
        skipped.append(r.run_id)
        continue
    raw_frames.append(frame)

raw = pd.concat(raw_frames, ignore_index=True)
has_rssi = raw.groupby('run_id')['rssi_estimated_distance_m'].apply(lambda s: s.notna().any())
print(f'Loaded {len(raw_frames)} runs, skipped {len(skipped)} empty runs.')
print(f'Runs with rssi_estimated_distance_m available: {int(has_rssi.sum())}/{len(has_rssi)}')
print(f'Raw rows: {len(raw):,}  |  row-level sybil rate: {raw["is_sybil"].mean():.4f}')
print(raw.groupby('row_type')['is_sybil'].mean().rename('sybil_rate_by_row_type'))
raw.head(3)


## 3. RSU-tier feature engineering: pooled tumbling windows

Windows are now tumbling groups of `W` **pooled rows** per `(run_id, rsu_id,
observed_claimed_id)` — every reporter's sighting of that claimed identity at
that RSU, chronologically, regardless of which vehicle reported it. This is the
structural difference from Approach 1 (whose windows were per single observer).
As in Approach 1, the trailing partial chunk is kept down to `min_beacons=1`
rather than dropped, for the same short-episode reason (type-3 attack).

Each window yields the same `n_beacons` / `n_beacons_norm` / `beacon_rate`
features as Approach 1, plus new **corroboration features** only meaningful
because multiple independent reporters are pooled here:

| Feature | Meaning |
|---|---|
| `n_reporters` | Distinct `reported_by_vehicle_id` values in the window — how many independent witnesses corroborate this claimed identity. |
| `real_id_agreement` | Share of rows agreeing with the plurality `observed_real_id` (1.0 = every reporter agrees on who this identity really is; lower = reporters contradict each other — a direct, multi-witness Sybil signal a single observer could never produce). |
| `self_third_party_mismatch` | 1 if a `self_report` row exists for this claimed identity in the window *and* its real id disagrees with what the third-party reporters say — i.e. the entity is telling the RSU one thing about itself while everyone else sees another. |
| `pos_spread` | Combined std-dev of reported `(bsm_x, bsm_y)` across third-party rows in the window — inconsistent reported positions across independent witnesses. |
| `rssi_gap` | Mean absolute gap between the vehicle-claimed `estimated_distance` and the RSU's own independent `rssi_estimated_distance_m` (when the schema has it) — a spoofed position should disagree with the RSU's own RF-based distance estimate. |

**Rate-normalization subtlety worth stating explicitly:** because rows are pooled
across `n_reporters` witnesses, the raw pooled arrival rate scales with
`n_reporters`, not with the single-vehicle 10 Hz nominal beaconing rate used in
Approach 1. `beacon_rate` is therefore divided by `n_reporters` before comparing
it to `R_NOM_DEFAULT`, so `T_behav` (§4) stays comparable across windows with
different numbers of corroborating witnesses.


In [ ]:
def build_rsu_windows(df: pd.DataFrame, window: int = 10, min_beacons: int = 1) -> pd.DataFrame:
    rows = []
    group_cols = ["run_id", "rsu_id", "observed_claimed_id"]
    for key, group in df.groupby(group_cols, sort=False):
        group = group.sort_values("time")
        n = len(group)
        for start in range(0, n, window):
            chunk = group.iloc[start:start + window]
            if len(chunk) < min_beacons:
                continue

            duration = max(chunk["time"].iloc[-1] - chunk["time"].iloc[0], 1e-6)
            n_chunk = len(chunk)
            beacon_rate = (n_chunk - 1) / duration if n_chunk > 1 else 0.0

            third_party = chunk[chunk["row_type"] == "third_party"]
            self_rows = chunk[chunk["row_type"] == "self_report"]

            n_reporters = int(chunk["reported_by_vehicle_id"].nunique())
            real_id_counts = chunk["observed_real_id"].value_counts()
            majority_real_id = real_id_counts.idxmax()
            real_id_agreement = float(real_id_counts.max() / n_chunk)
            n_real_ids = int(chunk["observed_real_id"].nunique())

            self_third_party_mismatch = 0
            if len(self_rows) > 0 and len(third_party) > 0:
                self_real_id = self_rows["observed_real_id"].iloc[0]
                tp_majority_real_id = third_party["observed_real_id"].value_counts().idxmax()
                self_third_party_mismatch = int(self_real_id != tp_majority_real_id)

            if len(third_party) > 1:
                pos_spread = float(np.hypot(third_party["bsm_x"].std(ddof=0),
                                             third_party["bsm_y"].std(ddof=0)))
            else:
                pos_spread = 0.0

            if len(third_party) > 0:
                token_valid_frac = float(third_party["token_valid"].mean())
                any_suspicion = int((third_party["suspicion_flags"].fillna(0).astype(int) != 0).any())
                rssi_gap = float((third_party["estimated_distance"]
                                   - third_party["rssi_estimated_distance_m"]).abs().mean())
                rsu_rssi_distance_mean = float(third_party["rssi_estimated_distance_m"].mean())
            else:
                token_valid_frac, any_suspicion = 1.0, 0
                rssi_gap, rsu_rssi_distance_mean = np.nan, np.nan

            rows.append({
                "run_id": key[0], "rsu_id": key[1], "observed_claimed_id": key[2],
                "observed_real_id": majority_real_id,
                "window_start": chunk["time"].iloc[0], "window_end": chunk["time"].iloc[-1],
                "n_beacons": n_chunk, "n_beacons_norm": n_chunk / window,
                "beacon_rate": beacon_rate,
                "n_reporters": n_reporters, "real_id_agreement": real_id_agreement,
                "n_real_ids": n_real_ids, "self_third_party_mismatch": self_third_party_mismatch,
                "pos_spread": pos_spread, "token_valid_frac": token_valid_frac,
                "any_suspicion": any_suspicion, "rssi_gap": rssi_gap,
                "rsu_rssi_distance_mean": rsu_rssi_distance_mean,
                "controller_id": chunk["controller_id"].iloc[0],
                "attack_type": chunk["attack_type"].iloc[0], "attack_name": chunk["attack_name"].iloc[0],
                "attack_percentage": chunk["attack_percentage"].iloc[0],
                "vehicle_count": chunk["vehicle_count"].iloc[0],
                "label": int(chunk["is_sybil"].max()),
            })
    return pd.DataFrame(rows)


WINDOW_W = 10
windows_raw = build_rsu_windows(raw, window=WINDOW_W)
print(f'RSU-tier windows (W={WINDOW_W}): {len(windows_raw):,}  |  positive rate: {windows_raw["label"].mean():.4f}')
windows_raw.head(3)


## 4. Trust-equation features, extended with RSU-tier corroboration

`T_RSSI`, `T_behav`, `T_hist`, `T_composite` reuse the exact same equations as
Approach 1 (Eqs 3.3, 3.22-3.25, including the same closed-form fix for the
`T`/`T_hist` circularity) — nothing about the thesis's analytic trust formula is
changed. What differs is *what feeds it*:

- `T_RSSI` (Eq 3.23) needs a co-location similarity `Φ_coloc` between different
  claimed identities' physical fingerprint at the same vantage point. Approach 1
  used per-beacon RSSI (dBm) as seen by one observing vehicle. At the RSU tier the
  more faithful fingerprint is `rssi_estimated_distance_m` — the RSU's own
  RF-based distance estimate, independent of any vehicle's self-report — compared
  across different claimed identities seen by the *same RSU* in overlapping
  windows. Where a run's schema lacks that column (36/79 runs, §2), the existing
  `phi_coloc` NaN-guard already falls back to a neutral similarity of 0, so this
  degrades gracefully rather than erroring.
- `T_behav` (Eq 3.24) uses the per-reporter-normalized `beacon_rate` from §3.

The new corroboration evidence (`n_reporters`, `real_id_agreement`,
`self_third_party_mismatch`, `pos_spread`) is **not** folded into the analytic `T`
formula — Eqs 3.22-3.25 don't define a corroboration term, and inventing
coefficients for one would be a fabrication the same way Approach 1 declined to
fabricate S1-S6 signature scores. Instead these are added as extra MLP input
features (§5) — the network, not a hand-picked equation, learns how much weight
RSU-level corroboration should get relative to the analytic trust score.


In [ ]:
R_NOM_DEFAULT = 10.0  # nominal per-reporter BSM broadcast rate, Hz (10 Hz beaconing, see docs/README.md)
N_REPORTERS_SCALE = 5.0   # reporters beyond this are treated as "fully corroborated" (n_reporters_norm caps at 1.0)
POS_SPREAD_SCALE = 50.0   # meters; position disagreement beyond this caps pos_spread_norm at 1.0


def phi_coloc(val_a: float, val_b: float, sigma_ch: float) -> float:
    # Eq 3.3 similarity kernel, reused unchanged from Approach 1.
    if not (np.isfinite(val_a) and np.isfinite(val_b)):
        return 0.0
    return float(np.exp(-((val_a - val_b) ** 2) / (2.0 * sigma_ch ** 2)))


def add_coloc_features(windows: pd.DataFrame, sigma_ch: float, gamma_co: float) -> pd.DataFrame:
    # Same overlap-window logic as Approach 1's add_coloc_features, grouped by
    # (run, RSU) instead of (run, observer vehicle), fingerprinting on
    # rsu_rssi_distance_mean instead of a per-beacon RSSI value.
    windows = windows.copy()
    coloc_mean = np.zeros(len(windows))
    coloc_max = np.zeros(len(windows))
    for _, idx in windows.groupby(["run_id", "rsu_id"]).groups.items():
        sub = windows.loc[idx]
        if len(sub) < 2:
            continue
        starts = sub["window_start"].to_numpy()
        ends = sub["window_end"].to_numpy()
        fingerprint = sub["rsu_rssi_distance_mean"].to_numpy()
        local_idx = sub.index.to_numpy()
        for a in range(len(sub)):
            overlaps = (starts <= ends[a]) & (ends >= starts[a])
            overlaps[a] = False
            if not overlaps.any():
                continue
            sims = [phi_coloc(fingerprint[a], fingerprint[b], sigma_ch) for b in np.nonzero(overlaps)[0]]
            pos = np.where(windows.index == local_idx[a])[0][0]
            coloc_mean[pos] = float(np.mean(sims))
            coloc_max[pos] = float(np.max(sims))
    windows["raw_coloc_mean"] = coloc_mean
    windows["raw_coloc_max"] = coloc_max
    windows["coloc_flag"] = (windows["raw_coloc_max"] > gamma_co).astype(int)
    return windows


def add_trust_scores(windows: pd.DataFrame, alpha: float, beta: float, gamma: float,
                      lam: float, mu: float, r_nom: float = R_NOM_DEFAULT) -> pd.DataFrame:
    # Eqs 3.22-3.25, identical closed-form resolution to Approach 1, keyed on (run, rsu, claimed_id).
    assert abs(alpha + beta + gamma - 1.0) < 1e-6, "alpha+beta+gamma must equal 1"
    windows = windows.sort_values(["run_id", "rsu_id", "observed_claimed_id", "window_start"]).copy()

    per_reporter_rate = windows["beacon_rate"] / windows["n_reporters"].clip(lower=1)
    windows["raw_rate_deviation"] = (per_reporter_rate - r_nom).abs() / r_nom
    windows["T_behav"] = np.exp(-lam * windows["raw_rate_deviation"])
    windows["T_RSSI"] = 1.0 - windows["raw_coloc_mean"]

    T_hist = np.zeros(len(windows))
    T_composite = np.zeros(len(windows))
    denom = 1.0 - gamma * (1.0 - mu)
    prev_T: Dict[Tuple, float] = {}
    for row_pos, (_, row) in enumerate(windows.iterrows()):
        key = (row["run_id"], row["rsu_id"], row["observed_claimed_id"])
        t_prev = prev_T.get(key, 0.5)  # bootstrap prior
        t_now = (alpha * row["T_RSSI"] + beta * row["T_behav"] + gamma * mu * t_prev) / denom
        t_now = float(np.clip(t_now, 0.0, 1.0))
        h_now = mu * t_prev + (1.0 - mu) * t_now
        T_composite[row_pos] = t_now
        T_hist[row_pos] = float(np.clip(h_now, 0.0, 1.0))
        prev_T[key] = t_now

    windows["T_hist"] = T_hist
    windows["T_composite"] = T_composite
    windows["n_reporters_norm"] = (windows["n_reporters"] / N_REPORTERS_SCALE).clip(upper=1.0)
    windows["pos_spread_norm"] = (windows["pos_spread"] / POS_SPREAD_SCALE).clip(upper=1.0)
    return windows.reset_index(drop=True)


# Default hyperparameters -- same status as Approach 1's DEFAULT_HP (none pinned in the
# thesis excerpt; grid-searched on validation in §10). sigma_ch is derived from the
# RSU's own RSSI-distance fingerprint instead of per-beacon RSSI dBm.
_sigma_ch = float(raw["rssi_estimated_distance_m"].std(skipna=True))
DEFAULT_HP = dict(alpha=0.4, beta=0.3, gamma=0.3, lam=1.0, mu=0.3,
                  sigma_ch=_sigma_ch if np.isfinite(_sigma_ch) and _sigma_ch > 0 else 10.0,
                  gamma_co=0.7)
print('Default hyperparameters:', {k: round(v, 4) if isinstance(v, float) else v for k, v in DEFAULT_HP.items()})

windows_raw = add_coloc_features(windows_raw, sigma_ch=DEFAULT_HP['sigma_ch'], gamma_co=DEFAULT_HP['gamma_co'])
windows = add_trust_scores(windows_raw, alpha=DEFAULT_HP['alpha'], beta=DEFAULT_HP['beta'],
                            gamma=DEFAULT_HP['gamma'], lam=DEFAULT_HP['lam'], mu=DEFAULT_HP['mu'])
print(f'Windows (W={WINDOW_W}): {len(windows):,}  |  positive rate: {windows["label"].mean():.4f}')
windows.head(3)


## 5. Assemble the RSU-tier feature table

Input vector (12 features): the same analytic core as Approach 1
(`T_RSSI`, `T_behav`, `T_hist`, `T_composite`, `token_valid_frac`,
`raw_rate_deviation`, `raw_coloc_mean`, `n_beacons_norm`) plus the four new
RSU-only corroboration features from §3 (`n_reporters_norm`, `real_id_agreement`,
`self_third_party_mismatch`, `pos_spread_norm`). The ablation study in §13
isolates how much the four new features actually contribute.


In [ ]:
FEATURE_COLS = [
    "T_RSSI", "T_behav", "T_hist", "T_composite",
    "token_valid_frac", "raw_rate_deviation", "raw_coloc_mean", "n_beacons_norm",
    "n_reporters_norm", "real_id_agreement", "self_third_party_mismatch", "pos_spread_norm",
]
META_COLS = [
    "run_id", "rsu_id", "observed_claimed_id", "observed_real_id", "controller_id",
    "attack_type", "attack_name", "attack_percentage", "vehicle_count",
    "window_start", "window_end", "label", "any_suspicion", "n_reporters", "n_real_ids",
]

dataset = windows[META_COLS + FEATURE_COLS].copy()
dataset["group_key"] = dataset["run_id"] + "__vid" + dataset["observed_real_id"].astype(str)
print(f'Final dataset: {len(dataset):,} samples, {len(FEATURE_COLS)} features, '
      f'{dataset["group_key"].nunique():,} unique (run, ground-truth-vehicle) groups.')
dataset.describe()[FEATURE_COLS]


## 6. Exploratory data analysis

As in Approach 1, attack types 1/5/6 are expected to contribute zero (or very
few) positive RSU-tier samples for the same structural reason (their Sybil
identity never surfaces as a spoofed third-party sighting). New here:
`n_reporters` and `real_id_agreement` distributions by label — if the
corroboration signal is doing real work, Sybil windows should show visibly lower
`real_id_agreement` than legitimate windows.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = dataset['label'].value_counts().sort_index()
axes[0].bar(['Legitimate', 'Sybil'], class_counts.values, color=['#4CAF50', '#F44336'])
axes[0].set_ylabel('Window samples')
axes[0].set_title(f'Overall class balance (n={len(dataset):,}, '
                   f'{100 * dataset["label"].mean():.1f}% positive)')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom')

per_type = dataset.groupby(['attack_type', 'attack_name']).agg(
    n_windows=('label', 'size'), n_positive=('label', 'sum'),
).reset_index()
per_type['positive_rate'] = per_type['n_positive'] / per_type['n_windows']
axes[1].bar(per_type['attack_name'], per_type['positive_rate'], color='#1f77b4')
axes[1].set_xticklabels(per_type['attack_name'], rotation=35, ha='right')
axes[1].set_ylabel('Positive (Sybil) window rate')
axes[1].set_title('Positive rate by attack type (pooled across percentages)')
plt.tight_layout()
plt.show()

display(per_type)


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=dataset, x='label', y='real_id_agreement', ax=axes[0], hue='label',
            palette={0: '#4CAF50', 1: '#F44336'}, legend=False)
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['Legit', 'Sybil'])
axes[0].set_title('Reporter identity agreement by label')
sns.boxplot(data=dataset, x='label', y='n_reporters', ax=axes[1], hue='label',
            palette={0: '#4CAF50', 1: '#F44336'}, legend=False)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['Legit', 'Sybil'])
axes[1].set_title('Number of corroborating reporters by label')
plt.tight_layout()
plt.show()

corr = dataset[FEATURE_COLS + ['label']].corr()
plt.figure(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Feature correlation (incl. label)')
plt.tight_layout()
plt.show()


## 7. Group-aware, stratified 70/15/15 split

Identical splitting strategy to Approach 1: grouped by `(run_id,
observed_real_id)` (the real vehicle, not the RSU or the reused small integer
claimed id), stratified within each `(attack_type, attack_percentage)` stratum,
so the same ground-truth vehicle never appears in both train and test.


In [ ]:
def group_stratified_split(df: pd.DataFrame, group_col: str, strata_cols: List[str],
                            train_frac=0.70, val_frac=0.15, seed=RNG_SEED) -> pd.Series:
    split = pd.Series(index=df.index, dtype=object)
    local_rng = np.random.default_rng(seed)
    for _, stratum in df.groupby(strata_cols):
        groups = stratum[group_col].unique()
        local_rng.shuffle(groups)
        n = len(groups)
        n_train = max(1, int(round(n * train_frac))) if n > 2 else n
        n_val = max(1, int(round(n * val_frac))) if n > 2 else 0
        n_train = min(n_train, n)
        n_val = min(n_val, n - n_train)
        train_groups = set(groups[:n_train])
        val_groups = set(groups[n_train:n_train + n_val])
        test_groups = set(groups[n_train + n_val:])
        mask = stratum[group_col].isin(train_groups)
        split.loc[stratum.index[mask]] = 'train'
        mask = stratum[group_col].isin(val_groups)
        split.loc[stratum.index[mask]] = 'val'
        mask = stratum[group_col].isin(test_groups)
        split.loc[stratum.index[mask]] = 'test'
    return split.fillna('train')


dataset['split'] = group_stratified_split(
    dataset, group_col='group_key', strata_cols=['attack_type', 'attack_percentage'],
)
split_summary = dataset.groupby('split').agg(
    n_samples=('label', 'size'), n_groups=('group_key', 'nunique'), positive_rate=('label', 'mean'),
).reindex(['train', 'val', 'test'])
display(split_summary)

overlap = (set(dataset.loc[dataset.split == 'train', 'group_key'])
           & set(dataset.loc[dataset.split == 'test', 'group_key']))
assert not overlap, f'Leakage: {len(overlap)} groups appear in both train and test'
print('No train/test group leakage confirmed.')


## 8. Baselines (to beat)

- **Baseline A — current C++ logic, replicated**, same as Approach 1: predict
  Sybil whenever any suspicion flag fired anywhere in the window (`any_suspicion`).
- **Baseline B — analytic composite T alone** (Eq 3.22, no MLP), threshold tuned
  on validation to maximize MCC, same as Approach 1.
- **Baseline C — corroboration alone (RSU-tier only, no analytic T)**: threshold
  `1 - real_id_agreement`, tuned on validation the same way. This isolates how
  much detection power comes from cross-reporter corroboration *by itself*,
  independent of the RSSI/behavior trust equation — the cleanest way to show what
  RSU-level pooling adds on top of Approach 1's evidence.


In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred) if len(set(y_true)) > 1 else 0.0,
        'fpr': fp / max(1, fp + tn),
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
    }


def tune_threshold(val_series: pd.Series, val_labels: pd.Series, higher_is_more_suspicious=True):
    best_thr, best_val_mcc = 0.5, -1.0
    for thr in np.linspace(0.05, 0.95, 37):
        score = val_series if higher_is_more_suspicious else (1.0 - val_series)
        pred = (score >= thr).astype(int)
        mcc = matthews_corrcoef(val_labels, pred) if val_labels.nunique() > 1 else 0.0
        if mcc > best_val_mcc:
            best_val_mcc, best_thr = mcc, thr
    return best_thr, best_val_mcc


train_df = dataset[dataset.split == 'train'].reset_index(drop=True)
val_df = dataset[dataset.split == 'val'].reset_index(drop=True)
test_df = dataset[dataset.split == 'test'].reset_index(drop=True)

# --- Baseline A: replicate scratch/Sybil-Developing-Improved.cc's trustScore logic ---
baseline_a_test_pred = test_df['any_suspicion'].to_numpy()
baseline_a_metrics = compute_metrics(test_df['label'].to_numpy(), baseline_a_test_pred)

# --- Baseline B: analytic composite T alone, threshold tuned on validation ---
best_thr_b, best_val_mcc_b = tune_threshold(1.0 - val_df['T_composite'], val_df['label'])
baseline_b_test_pred = ((1.0 - test_df['T_composite']) >= best_thr_b).astype(int).to_numpy()
baseline_b_metrics = compute_metrics(test_df['label'].to_numpy(), baseline_b_test_pred)

# --- Baseline C: RSU-tier corroboration alone, threshold tuned on validation ---
best_thr_c, best_val_mcc_c = tune_threshold(1.0 - val_df['real_id_agreement'], val_df['label'])
baseline_c_test_pred = ((1.0 - test_df['real_id_agreement']) >= best_thr_c).astype(int).to_numpy()
baseline_c_metrics = compute_metrics(test_df['label'].to_numpy(), baseline_c_test_pred)

print(f'Baseline A (current C++ logic replica): {baseline_a_metrics}')
print(f'Baseline B (analytic T alone, thr={best_thr_b:.3f}, val_mcc={best_val_mcc_b:.3f}): {baseline_b_metrics}')
print(f'Baseline C (corroboration alone, thr={best_thr_c:.3f}, val_mcc={best_val_mcc_c:.3f}): {baseline_c_metrics}')


## 9. Compact MLP RSU Trust Analyzer — identical architecture to Approach 1

Same `Linear(n_features, 32) -> ReLU -> Dropout -> Linear(32, 16) -> ReLU ->
Linear(16, 1)` design (only `n_features` differs: 12 here vs 8 in Approach 1).
Keeping the architecture identical isolates the comparison between the two
approaches to *what evidence they see and how they aggregate*, not to
architecture differences.


In [ ]:
class TrustMLP(nn.Module):
    def __init__(self, n_features: int, hidden1: int = 32, hidden2: int = 16, dropout: float = 0.3):
        super().__init__()
        self.fc1 = nn.Linear(n_features, hidden1)
        self.act1 = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.act2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden2, 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h1 = self.drop(self.act1(self.fc1(x)))
        phi_trust = self.act2(self.fc2(h1))  # deployed output, Eq 3.19's phi_trust(v_i) analogue at RSU tier
        logit = self.fc3(phi_trust).squeeze(-1)
        return logit, phi_trust


n_features = len(FEATURE_COLS)
_sanity = TrustMLP(n_features)
_x = torch.randn(4, n_features)
_logit, _phi = _sanity(_x)
print(f'MLP sanity check: input {tuple(_x.shape)} -> logit {tuple(_logit.shape)}, '
      f'phi_trust {tuple(_phi.shape)} (expected (*, 16))')


## 10. Single-hop federated training: RSU client &rarr; controller

Each **FL client** = one `(run_id, rsu_id)` pair — the RSU pools every vehicle
report it receives locally, and trains on that pooled window table. There is no
mid-tier grouping: the client already is the entity Approach 1's Eq 3.27 would
have aggregated *into*. So the round loop collapses to:

sample a fraction of RSU clients &rarr; each trains locally for `local_epochs`
with a FedProx proximal term &rarr; the client's weight delta gets Gaussian DP
noise before upload &rarr; **controller tier**: Byzantine-robust trimmed mean
directly across the sampled RSU-client updates (Eq 3.28), dropping the `trim_k`
highest/lowest-norm updates.

**Simplification, stated explicitly (differs from Approach 1 here):** Eq 3.28 is
an *unweighted* average after trimming, unlike Eq 3.27's sample-size weighting —
so unlike Approach 1's RSU-tier step, this aggregation does **not** weight RSU
clients by how many windows they contributed. This is a direct, literal reading
of Eq 3.28 now that the RSU is the atomic client, and it also matches the
Byzantine-robustness goal: weighting by sample count would let a highly active
(or attack-flooded) RSU dominate the trimmed mean.

**Same single-controller caveat as Approach 1 §2**: this v10-v40 sweep has one
global controller, so "controller tier" here means trimming across *all* sampled
RSU-clients from *all* runs each round, not per-controller sub-groups. With only
5-10 RSUs per run (checked directly against the logs), the total client pool per
round is small — `trim_k` is kept at 1 for this reason (§11).

**A collapse found while building this, and the fix.** 61% of RSU clients
(364/589 in the train split) are *label-degenerate* — every window they hold is
legitimate, because they only ever serve attack types 0/5/6 (baseline, malicious
RSU, malicious controller), which structurally never produce a positive
RSU-tier window (same reason as Approach 1's finding #7). Pooling reporters
(§3) also makes each client's local sample count larger than Approach 1's
per-vehicle clients (mean ~16-30 windows here vs. a handful there), so a
label-degenerate client's local training converges *confidently* toward
"always predict legitimate" in just `local_epochs`. With **uniform** random
client sampling and this large a degenerate-client majority, an unweighted
trimmed mean across a round's sampled clients is dominated by these confident,
uniformly-negative local updates — the global model collapsed to predicting
legitimate for every validation window within a single round (verified
directly: validation recall hit exactly 0 after round 1, and predicted
probabilities monotonically shrank toward 0 every round after). This is a
sharper version of Approach 1's own §9 finding, not a different bug: same
root cause (severe label skew across FL clients), worse here because pooling
concentrates it into fewer, larger, more one-sided clients.

**Fix: label-stratified client sampling.** Each round, clients are drawn from
two pools — those holding at least one Sybil window (`positive_keys`) and those
with none (`other_keys`) — and a fixed share `POSITIVE_CLIENT_FRAC` of each
round's selection is guaranteed to come from `positive_keys` (topped up from
`other_keys` for the rest), instead of sampling uniformly over all clients. This
is a standard response to label skew across FL clients in the federated-learning
literature, not a hand-tuned trick to inflate a metric: it only changes *which*
clients are seen each round, never which data a client is allowed to see, and it
does not touch DP noise, FedProx, or the trimmed-mean aggregation itself.


In [ ]:
def clone_state_dict(sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k: v.clone() for k, v in sd.items()}


POSITIVE_CLIENT_FRAC = 0.6  # share of each round's selected clients guaranteed to hold >=1 Sybil window (see §10 fix)


def build_client_cache(df: pd.DataFrame, feature_cols: List[str]) -> Dict[Tuple, Tuple]:
    # One client per (run_id, rsu_id) -- the RSU IS the client, no vehicle sub-tier.
    cache = {}
    for key, cdf in df.groupby(['run_id', 'rsu_id'], sort=False):
        if len(cdf) < 2:
            continue
        X = torch.tensor(cdf[feature_cols].to_numpy(dtype=np.float32))
        y = torch.tensor(cdf['label'].to_numpy(dtype=np.float32))
        cache[key] = (X, y)
    return cache


def split_clients_by_label(client_cache: Dict[Tuple, Tuple], client_keys: List[Tuple]) -> Tuple[List, List]:
    positive_keys = [k for k in client_keys if client_cache[k][1].sum().item() > 0]
    other_keys = [k for k in client_keys if client_cache[k][1].sum().item() == 0]
    return positive_keys, other_keys


def local_train(global_state, X_t, y_t, epochs, lr, fedprox_mu, batch_size, n_features):
    model = TrustMLP(n_features)
    model.load_state_dict(global_state)
    global_params = clone_state_dict(global_state)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    n = len(y_t)
    idx_all = np.arange(n)
    for _ in range(epochs):
        rng.shuffle(idx_all)
        for start in range(0, n, batch_size):
            idx = idx_all[start:start + batch_size]
            xb, yb = X_t[idx], y_t[idx]
            opt.zero_grad()
            logits, _ = model(xb)
            loss = criterion(logits, yb)
            prox = sum(((p - global_params[name]) ** 2).sum() for name, p in model.named_parameters())
            loss = loss + (fedprox_mu / 2.0) * prox
            loss.backward()
            opt.step()
    return model.state_dict(), n


def state_dict_delta_norm(sd: Dict, ref_sd: Dict) -> float:
    total = 0.0
    for key in sd:
        total += (sd[key] - ref_sd[key]).float().pow(2).sum().item()
    return total ** 0.5


def trimmed_mean_state_dict(state_dicts: List[Dict], ref_state: Dict, trim_k: int) -> Dict[str, torch.Tensor]:
    # Eq 3.28: unweighted trimmed mean, applied directly to RSU-client updates
    # (no sample-size weighting -- see the §10 simplification note).
    n = len(state_dicts)
    if n <= 2 * trim_k:
        keep = list(range(n))  # too few RSU clients this round to trim safely
    else:
        norms = [state_dict_delta_norm(sd, ref_state) for sd in state_dicts]
        keep = list(np.argsort(norms)[trim_k:n - trim_k])
    out = {}
    for key in state_dicts[0]:
        out[key] = torch.stack([state_dicts[i][key].float() for i in keep], dim=0).mean(dim=0)
    return out


def run_fl_round(global_state, client_cache, positive_keys, other_keys, hp, n_features,
                  selection_fraction=0.3, min_clients=8, positive_frac=POSITIVE_CLIENT_FRAC):
    total_clients = len(positive_keys) + len(other_keys)
    n_select = min(total_clients, max(min_clients, int(math.ceil(total_clients * selection_fraction))))
    n_pos = min(len(positive_keys), max(1, int(round(n_select * positive_frac))) if positive_keys else 0)
    n_other = min(len(other_keys), n_select - n_pos)
    idx_pos = rng.choice(len(positive_keys), size=n_pos, replace=False) if n_pos > 0 else np.array([], dtype=int)
    idx_other = rng.choice(len(other_keys), size=n_other, replace=False) if n_other > 0 else np.array([], dtype=int)
    selected_keys = [positive_keys[i] for i in idx_pos] + [other_keys[i] for i in idx_other]

    noised_states = []
    total_samples = 0
    for key in selected_keys:
        X_t, y_t = client_cache[key]
        local_state, n = local_train(global_state, X_t, y_t, hp['local_epochs'], hp['local_lr'],
                                      hp['fedprox_mu'], hp['batch_size'], n_features)
        noised = {}
        for k_, v in local_state.items():
            delta = v - global_state[k_]
            noisy_delta = delta + torch.randn(delta.shape) * hp['dp_sigma']
            noised[k_] = global_state[k_] + noisy_delta
        noised_states.append(noised)
        total_samples += n

    if not noised_states:
        return global_state, 0

    new_global = trimmed_mean_state_dict(noised_states, global_state, trim_k=hp['trim_k'])
    return new_global, total_samples


def evaluate_state(state, df, feature_cols, n_features):
    model = TrustMLP(n_features, dropout=0.0)
    model.load_state_dict(state)
    model.eval()
    X = torch.tensor(df[feature_cols].to_numpy(dtype=np.float32))
    with torch.no_grad():
        logits, phi = model(X)
        probs = torch.sigmoid(logits).numpy()
    pred = (probs >= 0.5).astype(int)
    metrics = compute_metrics(df['label'].to_numpy(), pred)
    return metrics, probs, phi.numpy()


def train_fl(train_part, val_part, feature_cols, hp, n_features,
             max_rounds=15, patience=5, min_delta=1e-3, verbose=False,
             selection_fraction=0.3, min_clients=8):
    torch.manual_seed(RNG_SEED)
    model0 = TrustMLP(n_features, dropout=hp.get('dropout', 0.3))
    global_state = clone_state_dict(model0.state_dict())
    client_cache = build_client_cache(train_part, feature_cols)
    client_keys = list(client_cache.keys())
    positive_keys, other_keys = split_clients_by_label(client_cache, client_keys)

    history = []
    best_mcc, best_state, rounds_no_improve = -1.0, global_state, 0
    for rnd in range(1, max_rounds + 1):
        global_state, n_trained = run_fl_round(global_state, client_cache, positive_keys, other_keys, hp, n_features,
                                                selection_fraction=selection_fraction, min_clients=min_clients)
        val_metrics, _, _ = evaluate_state(global_state, val_part, feature_cols, n_features)
        history.append({'round': rnd, 'val_mcc': val_metrics['mcc'], 'val_f1': val_metrics['f1'],
                         'val_accuracy': val_metrics['accuracy'], 'n_trained': n_trained})
        if val_metrics['mcc'] > best_mcc + min_delta:
            best_mcc, best_state, rounds_no_improve = val_metrics['mcc'], clone_state_dict(global_state), 0
        else:
            rounds_no_improve += 1
        if verbose and (rnd == 1 or rnd % 5 == 0 or rounds_no_improve >= patience):
            print(f'  round={rnd:3d}  val_mcc={val_metrics["mcc"]:.4f}  best={best_mcc:.4f}  '
                  f'no_improve={rounds_no_improve}')
        if rounds_no_improve >= patience:
            break
    return best_state, pd.DataFrame(history), best_mcc


n_rsu_clients = dataset.groupby(['run_id', 'rsu_id']).ngroups
print(f'Total RSU clients available across all runs: {n_rsu_clients} '
      f'(compare to hundreds of vehicle clients in Approach 1 -- much smaller client pool per round).')

# Smoke test: a handful of rounds on the real train/val split, default hyperparameters.
_smoke_hp = dict(dropout=0.1, fedprox_mu=0.001, dp_sigma=0.02, local_lr=0.01, batch_size=32,
                  local_epochs=2, trim_k=1)
import time as _time
_t0 = _time.time()
_state, _hist, _mcc = train_fl(train_df, val_df, FEATURE_COLS, _smoke_hp, n_features,
                                 max_rounds=20, patience=6, verbose=True,
                                 selection_fraction=0.5, min_clients=5)
print(f'Smoke test: {len(_hist)} rounds in {_time.time() - _t0:.1f}s, best val MCC so far = {_mcc:.4f}')


## 11. Hyperparameter search

Same reduced-grid approach as Approach 1, for the same reason (this is a ~small
sample, not the intended final dataset). `min_clients` is lower than Approach 1's
(5 vs 15) because there are only 5-10 RSU clients per run in total — Approach 1
could afford to demand 15 vehicle clients per round out of hundreds available.
`trim_k` is fixed at 1 rather than swept, since `2*trim_k` must stay well below
the total client count per round (§10).


In [ ]:
FULL_GRID = {  # reference only -- mirrors Approach 1's design-brief hyperparameter table
    'dropout': [0.1, 0.2, 0.3, 0.5],
    'fedprox_mu': [0.001, 0.01, 0.1],
    'dp_sigma': [0.05, 0.1, 0.2],
    'local_lr': [0.001, 0.005, 0.01, 0.05],
    'batch_size': [16, 32, 64],
    'local_epochs': [1, 3, 5],
    'trim_k': [1],
}

REDUCED_GRID = {  # actually executed in this notebook (see note above)
    'dropout': [0.1],
    'fedprox_mu': [0.001, 0.01],
    'dp_sigma': [0.02],
    'local_lr': [0.01],
    'batch_size': [32],
    'local_epochs': [2],
    'trim_k': [1],
}
N_CV_SPLITS = 2
CV_MAX_ROUNDS = 10
CV_PATIENCE = 4


def make_group_kfold(df: pd.DataFrame, group_col: str, n_splits: int, seed=RNG_SEED):
    groups = df[group_col].unique()
    local_rng = np.random.default_rng(seed)
    local_rng.shuffle(groups)
    folds = np.array_split(groups, n_splits)
    for i in range(n_splits):
        held_out = set(folds[i])
        held_mask = df[group_col].isin(held_out)
        yield df[~held_mask].reset_index(drop=True), df[held_mask].reset_index(drop=True)


import itertools

grid_keys = list(REDUCED_GRID.keys())
grid_results = []
for combo in itertools.product(*REDUCED_GRID.values()):
    hp = dict(zip(grid_keys, combo))
    fold_mccs = []
    for fold_train, fold_val in make_group_kfold(train_df, 'group_key', N_CV_SPLITS):
        _, _, fold_best_mcc = train_fl(fold_train, fold_val, FEATURE_COLS, hp, n_features,
                                        max_rounds=CV_MAX_ROUNDS, patience=CV_PATIENCE,
                                        selection_fraction=0.5, min_clients=5)
        fold_mccs.append(fold_best_mcc)
    grid_results.append({**hp, 'cv_mean_mcc': float(np.mean(fold_mccs)), 'cv_std_mcc': float(np.std(fold_mccs))})

grid_df = pd.DataFrame(grid_results).sort_values('cv_mean_mcc', ascending=False).reset_index(drop=True)
display(grid_df)
INT_HP_KEYS = {'batch_size', 'local_epochs', 'trim_k'}
best_hp = {k: (int(grid_df.iloc[0][k]) if k in INT_HP_KEYS else float(grid_df.iloc[0][k])) for k in grid_keys}
print('Selected hyperparameters (highest CV mean val MCC):', best_hp)


## 12. Final training run + FL convergence


In [ ]:
final_state, final_history, final_best_val_mcc = train_fl(
    train_df, val_df, FEATURE_COLS, best_hp, n_features,
    max_rounds=30, patience=8, verbose=True, selection_fraction=0.5, min_clients=5,
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(final_history['round'], final_history['val_mcc'], marker='o', color='#1f77b4', label='Validation MCC')
best_round = final_history.loc[final_history['val_mcc'].idxmax(), 'round']
ax.axvline(best_round, color='#F44336', linestyle='--', label=f'Best round ({int(best_round)})')
ax.set_xlabel('FL round')
ax.set_ylabel('Validation MCC')
ax.set_title('RSU-tier single-hop FL convergence (FedProx + DP noise + controller trimmed mean)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Best validation MCC: {final_best_val_mcc:.4f} at round {int(best_round)}')


## 13. Test-set evaluation: four-way comparison

Baseline A (current C++ logic) vs. Baseline B (analytic T alone) vs. Baseline C
(RSU corroboration alone) vs. the trained MLP RSU Trust Analyzer, all on the
untouched test split. As in Approach 1, read these as an infrastructure
demonstration on a small, non-IID sample, not final model performance.


In [ ]:
mlp_test_metrics, mlp_test_probs, mlp_test_phi = evaluate_state(final_state, test_df, FEATURE_COLS, n_features)

comparison = pd.DataFrame([
    {'model': 'Baseline A (current C++ logic)', **baseline_a_metrics},
    {'model': 'Baseline B (analytic T alone)', **baseline_b_metrics},
    {'model': 'Baseline C (RSU corroboration alone)', **baseline_c_metrics},
    {'model': 'MLP RSU Trust Analyzer (this notebook)', **mlp_test_metrics},
]).set_index('model')
display(comparison[['accuracy', 'precision', 'recall', 'f1', 'mcc', 'fpr']].round(4))

cm = confusion_matrix(test_df['label'], (mlp_test_probs >= 0.5).astype(int), labels=[0, 1])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Legit', 'Sybil'],
            yticklabels=['Legit', 'Sybil'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('MLP RSU Trust Analyzer — test confusion matrix')

fpr_b, tpr_b, _ = roc_curve(test_df['label'], 1.0 - test_df['T_composite'])
fpr_c, tpr_c, _ = roc_curve(test_df['label'], 1.0 - test_df['real_id_agreement'])
fpr_mlp, tpr_mlp, _ = roc_curve(test_df['label'], mlp_test_probs)
axes[1].plot(fpr_b, tpr_b, label=f'Baseline B (AUC={auc(fpr_b, tpr_b):.3f})', color='#ff7f0e')
axes[1].plot(fpr_c, tpr_c, label=f'Baseline C (AUC={auc(fpr_c, tpr_c):.3f})', color='#9467bd')
axes[1].plot(fpr_mlp, tpr_mlp, label=f'MLP (AUC={auc(fpr_mlp, tpr_mlp):.3f})', color='#1f77b4')
axes[1].scatter([baseline_a_metrics['fpr']], [baseline_a_metrics['recall']], color='#4CAF50',
                s=80, zorder=5, label='Baseline A (single point, no score)')
axes[1].plot([0, 1], [0, 1], linestyle=':', color='grey')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC — test split')
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
per_type_eval = test_df.copy()
per_type_eval['mlp_pred'] = (mlp_test_probs >= 0.5).astype(int)

rows = []
for (atype, aname), g in per_type_eval.groupby(['attack_type', 'attack_name']):
    row = {'attack_type': atype, 'attack_name': aname, 'n_windows': len(g), 'n_positive': int(g['label'].sum())}
    if g['label'].nunique() > 1:
        row['mlp_mcc'] = matthews_corrcoef(g['label'], g['mlp_pred'])
        row['baseline_a_mcc'] = matthews_corrcoef(g['label'], g['any_suspicion'])
    else:
        row['mlp_mcc'] = np.nan
        row['baseline_a_mcc'] = np.nan
    rows.append(row)

per_type_test = pd.DataFrame(rows)
display(per_type_test)


## 14. Ablation: contribution of each feature, new corroboration features included

Retrains with one feature ablated at a time. The four new RSU-only features
(`n_reporters_norm`, `real_id_agreement`, `self_third_party_mismatch`,
`pos_spread_norm`) are the ones this notebook adds over Approach 1 — this is the
direct evidence for whether pooling RSU-level observations actually improves the
trust score, or whether the analytic `T` features alone already capture
everything.

**Read this table with real skepticism, for a reason stated up front rather than
buried:** on this sample, single-run test MCC swings widely (from ~0 to ~0.58)
depending on which one feature is dropped, with everything else held fixed.
That spread is larger than a clean per-feature contribution signal should
produce — it is the same small-sample/severe-client-imbalance FL instability
documented in §10, showing up again here because each ablation retrains FL from
scratch with only one dataset's worth of randomness (one seed, ~10-30 rounds).
Treat the ranking below as directional, not as a definitive feature-importance
result — exactly the same caveat this notebook (and Approach 1) applies to every
other MCC value on this sample. A trustworthy ablation needs averaging over
multiple seeds and the larger intended dataset, not attempted here.


In [ ]:
ABLATION_ROUNDS, ABLATION_PATIENCE = 12, 5
ablation_targets = ['T_RSSI', 'T_behav', 'T_hist', 'T_composite', 'token_valid_frac',
                     'raw_rate_deviation', 'raw_coloc_mean', 'n_beacons_norm',
                     'n_reporters_norm', 'real_id_agreement', 'self_third_party_mismatch', 'pos_spread_norm']

ablation_rows = [{'ablated': '(none -- full feature set)',
                   'test_mcc': mlp_test_metrics['mcc'], 'n_features': len(FEATURE_COLS)}]

for feat in ablation_targets:
    cols = [c for c in FEATURE_COLS if c != feat]
    state, _, _ = train_fl(train_df, val_df, cols, best_hp, len(cols),
                            max_rounds=ABLATION_ROUNDS, patience=ABLATION_PATIENCE,
                            selection_fraction=0.5, min_clients=5)
    metrics, _, _ = evaluate_state(state, test_df, cols, len(cols))
    ablation_rows.append({'ablated': feat, 'test_mcc': metrics['mcc'], 'n_features': len(cols)})

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df)

new_rsu_features = {'n_reporters_norm', 'real_id_agreement', 'self_third_party_mismatch', 'pos_spread_norm'}
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#4CAF50'] + ['#F44336' if f not in new_rsu_features else '#9467bd' for f in ablation_targets]
ax.barh(ablation_df['ablated'], ablation_df['test_mcc'], color=colors)
ax.set_xlabel('Test MCC')
ax.set_title('Ablation: test MCC with each feature removed\n(purple = new RSU-tier corroboration feature)')
plt.tight_layout()
plt.show()


## 15. Window-size sensitivity (W in {5, 10, 20})

Same purpose as Approach 1 §14: isolate the effect of `W` on the analytic
composite `T` alone (Baseline B), rebuilt at each window size, without paying for
a full MLP retrain per size.


In [ ]:
window_sensitivity_rows = []
for w in (5, 10, 20):
    w_raw = build_rsu_windows(raw, window=w)
    w_raw = add_coloc_features(w_raw, sigma_ch=DEFAULT_HP['sigma_ch'], gamma_co=DEFAULT_HP['gamma_co'])
    w_full = add_trust_scores(w_raw, alpha=DEFAULT_HP['alpha'], beta=DEFAULT_HP['beta'],
                                gamma=DEFAULT_HP['gamma'], lam=DEFAULT_HP['lam'], mu=DEFAULT_HP['mu'])
    w_dataset = w_full[META_COLS + FEATURE_COLS].copy()
    w_dataset['group_key'] = w_dataset['run_id'] + '__vid' + w_dataset['observed_real_id'].astype(str)
    w_dataset['split'] = group_stratified_split(w_dataset, 'group_key', ['attack_type', 'attack_percentage'])

    w_val = w_dataset[w_dataset.split == 'val']
    w_test = w_dataset[w_dataset.split == 'test']
    best_w_thr, best_w_val_mcc = tune_threshold(1.0 - w_val['T_composite'], w_val['label'])
    test_pred = ((1.0 - w_test['T_composite']) >= best_w_thr).astype(int)
    test_mcc = matthews_corrcoef(w_test['label'], test_pred) if w_test['label'].nunique() > 1 else 0.0
    window_sensitivity_rows.append({'W': w, 'n_windows': len(w_dataset), 'positive_rate': w_dataset['label'].mean(),
                                     'val_mcc': best_w_val_mcc, 'test_mcc': test_mcc})

window_sensitivity_df = pd.DataFrame(window_sensitivity_rows)
display(window_sensitivity_df)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(window_sensitivity_df['W'], window_sensitivity_df['test_mcc'], marker='o', color='#1f77b4')
ax.set_xlabel('Window size W (pooled rows)')
ax.set_ylabel('Test MCC (Baseline B, analytic T only)')
ax.set_title('Window-size sensitivity — RSU tier')
ax.set_xticks([5, 10, 20])
plt.tight_layout()
plt.show()


## 16. Approach 1 vs. Approach 2 — what actually differs

Both notebooks share the same data source, ground-truth rule, MLP architecture,
FedProx/DP training machinery, and the same v10-v40 sample-data caveat. What
differs is client granularity and evidence:

| | Approach 1 (vehicle-tier) | Approach 2 (RSU-tier, this notebook) |
|---|---|---|
| FL client | one vehicle | one RSU |
| # clients per run | up to `vehicle_count` | 5-10 (checked directly against the logs) |
| Aggregation hops | 2 (vehicle &rarr; RSU, Eq 3.27; RSU &rarr; SDN, Eq 3.28) | 1 (RSU &rarr; controller, Eq 3.28 only) |
| Evidence per window | one observer's view of one claimed identity | every reporter's view of one claimed identity, pooled |
| Can see cross-reporter disagreement? | No — structurally impossible with one observer | Yes — `real_id_agreement`, `self_third_party_mismatch`, `pos_spread` |
| DP noise granularity | noise added per-vehicle update, many small updates | noise added per-RSU update, fewer, larger updates |
| Byzantine trimming | trims across many (typically dozens+) RSU-level averages already smoothed by Eq 3.27 | trims directly across a handful of raw RSU-client updates — sensitive to how big `trim_k` is relative to the small client count (§10) |

The practical takeaway for which to prefer in deployment is not decided by this
notebook — it depends on whether the richer per-window evidence here (§14
ablation) outweighs having far fewer, larger FL clients to trim against. Both
should be re-evaluated once the intended larger dataset exists (Approach 1 §16,
mismatch #4).


## 17. Export

Saves the trained RSU-tier MLP weights and every calibrated hyperparameter to
`sybil-attack/fl/trust_analyzer/`, as JSON and as a C++ header, in the same
format as Approach 1's export (`export_weights_json` / `export_cpp_header`) so
both analyzers can be loaded the same way from the simulator. Filenames are
prefixed `rsu_` to avoid clobbering Approach 1's `vehicle_trust_mlp_weights.json`.
Wiring either into `sybil_types.h` / `Sybil-Developing-Improved.cc` remains a
follow-up implementation task, not done in either notebook.


In [ ]:
def export_weights_json(state: Dict[str, torch.Tensor], hp: Dict, trust_hp: Dict,
                          feature_cols: List[str], path: Path) -> None:
    payload = {
        'feature_columns': feature_cols,
        'model_hyperparameters': {k: (v if not isinstance(v, (np.floating, np.integer)) else v.item())
                                   for k, v in hp.items()},
        'trust_equation_hyperparameters': trust_hp,
        'layers': {name: tensor.detach().numpy().tolist() for name, tensor in state.items()},
    }
    path.write_text(json.dumps(payload, indent=2))


def export_cpp_header(state: Dict[str, torch.Tensor], hp: Dict, trust_hp: Dict,
                        feature_cols: List[str], path: Path) -> None:
    lines = [
        '// Generated by sybil-attack/notebooks/rsu_trust_score_model.ipynb',
        '// RSU-tier Trust Analyzer -- 3-layer MLP, single-hop RSU-client FL (see docs/Sybil_attack_project.pdf',
        '// Eqs 3.3, 3.20, 3.22-3.25, 3.28)',
        '// NOT yet wired into sybil_types.h / Sybil-Developing-Improved.cc -- see notebook scope note.',
        '#pragma once', '',
        '// Feature order (must match at inference time):',
    ]
    for i, col in enumerate(feature_cols):
        lines.append(f'//   {i}: {col}')
    lines.append('')
    for key, name in [('alpha', 'kAlpha'), ('beta', 'kBeta'), ('gamma', 'kGamma'), ('lam', 'kLambda'),
                       ('mu', 'kMu'), ('sigma_ch', 'kSigmaCh'), ('gamma_co', 'kGammaCo')]:
        lines.append(f'static const double {name} = {trust_hp[key]:.12g};')
    lines.append('')
    for layer_name, tensor in state.items():
        arr = tensor.detach().numpy()
        c_name = 'k' + layer_name.replace('.', '_')
        if arr.ndim == 1:
            values = ', '.join(f'{v:.10g}' for v in arr)
            lines.append(f'static const double {c_name}[{arr.shape[0]}] = {{{values}}};')
        else:
            rows = []
            for row in arr:
                rows.append('{' + ', '.join(f'{v:.10g}' for v in row) + '}')
            lines.append(f'static const double {c_name}[{arr.shape[0]}][{arr.shape[1]}] = {{{", ".join(rows)}}};')
    path.write_text('\n'.join(lines) + '\n')


trust_hp_export = {k: v for k, v in DEFAULT_HP.items()}
export_weights_json(final_state, best_hp, trust_hp_export, FEATURE_COLS,
                     EXPORT_DIR / 'rsu_trust_mlp_weights.json')
export_cpp_header(final_state, best_hp, trust_hp_export, FEATURE_COLS,
                    EXPORT_DIR / 'rsu_trust_analyzer.h')
print(f'Exported to {EXPORT_DIR / "rsu_trust_mlp_weights.json"} and {EXPORT_DIR / "rsu_trust_analyzer.h"}')


## 18. Summary

**What this notebook is.** The RSU-tier counterpart to
`vehicle_trust_score_model.ipynb`: single-hop federated training (RSU client
&rarr; controller, Eq 3.28 only) over pooled multi-reporter observations, with
four new corroboration features (`n_reporters_norm`, `real_id_agreement`,
`self_third_party_mismatch`, `pos_spread_norm`) added specifically because an RSU
sees every vehicle's report of a claimed identity, not just one observer's.

**What the numbers in this run are not.** Same caveat as Approach 1: the
`evaluation_runs/` v10-v40 sweep is a sample, not the intended dataset, and
several attack types contribute few or zero positive RSU-tier windows by
construction. Every result above is "the pipeline runs correctly and produces
sane, explicable output," not a final trust-score result.

**Next steps (outside this notebook):**
1. Generate the intended larger dataset (Approach 1 §16, next step 1) — this
   notebook needs no code changes beyond pointing `EVAL_RUNS_DIR` at it.
2. Re-run the ablation (§14) on the larger dataset to get a trustworthy answer on
   how much the new corroboration features are worth.
3. Decide, using both notebooks' results on real data, whether to deploy the
   vehicle-tier analyzer, the RSU-tier analyzer, or ensemble both (e.g. feed
   Approach 1's `φ_trust(v_i)` in as an extra input feature to Approach 2's MLP,
   or vice versa) — not attempted here to keep the two approaches independently
   comparable.
4. Port the exported RSU-tier weights (`rsu_trust_mlp_weights.json` /
   `rsu_trust_analyzer.h`) into the simulator alongside Approach 1's, once a
   deployment decision is made.
